# Student Information

| Field       | Details              |
|-------------|----------------------|
| **Name**    | Nimesh Timalsina     |
| **Roll No** | 26                   |

---

# Squeeze-and-Excitation (SE) Attention in Object Detection

## Assignment Overview

This notebook investigates whether **Squeeze-and-Excitation (SE) channel attention** can improve object detection performance when inserted into a standard CNN backbone. Two detectors are trained and compared under identical conditions:

| Model | Description |
|-------|-------------|
| **Baseline** | Faster R-CNN with a ResNet-50-FPN backbone (no attention) |
| **SE Model**  | Faster R-CNN with the same backbone, but SE blocks inserted inside every ResNet Bottleneck block |

---

## Objectives

1. Understand how SE channel attention works and where it fits in a detection pipeline.
2. Train both models from the same pretrained ResNet-50 weights to isolate the effect of SE.
3. Compare accuracy (mAP, Precision, Recall) and computational cost (training time, FPS).
4. Draw conclusions about when SE attention is beneficial for object detection.

---

## Dataset — Penn-Fudan Pedestrian

- **Source:** University of Pennsylvania / Fudan University
- **Task:** Pedestrian detection (binary: background vs. person)
- **Size:** 170 images with pixel-level segmentation masks
- **Split used:** 70% train / 15% val / 15% test (119 / 25 / 26 images)
- **Why this dataset?** Small enough to train fully in one session on a single GPU while still being a real detection benchmark.

> You can substitute Pascal VOC, COCO, or a custom dataset without changing any of the SE architecture code.

---

## Architecture Summary

```
Input Image
     │
 ResNet-50 Backbone (with or without SE blocks)
     │
  FPN (Feature Pyramid Network)  ←── multi-scale features
     │
  RPN (Region Proposal Network)  ←── object proposals
     │
  RoI Align + Detection Head     ←── classify & refine boxes
     │
 Predictions (boxes, labels, scores)
```

**SE block position:** inserted after the last BN in each Bottleneck block, before the residual addition.

```
Bottleneck:
  conv1 → BN → ReLU
  conv2 → BN → ReLU
  conv3 → BN
              ↓
           SE Block  (squeeze → excite → scale)
              ↓
  + residual → ReLU
```

In [ ]:
# Install dependencies (run once)
!pip -q install torchmetrics matplotlib pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 20.5 MB/s eta 0:00:00


In [ ]:
import os
import time
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

import torchvision
from torchvision.models import ResNet50_Weights
from torchvision.models.resnet import ResNet, Bottleneck, resnet50
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.backbone_utils import BackboneWithFPN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import box_iou, MultiScaleRoIAlign
from torchmetrics.detection.mean_ap import MeanAveragePrecision

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)

torch: 2.11.0+cu128
torchvision: 0.26.0+cu128


In [ ]:
# Reproducibility and training settings
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

device = get_device()
print("Device:", device)

DATA_ROOT = Path("./data")
NUM_EPOCHS = 10
BATCH_SIZE = 2
NUM_WORKERS = 2
LR = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
SCORE_THRESH = 0.05
IOU_THRESH = 0.5

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-6

In [ ]:
# Download Penn-Fudan Pedestrian dataset if needed
DATA_DIR = DATA_ROOT / "PennFudanPed"
ZIP_PATH = DATA_ROOT / "PennFudanPed.zip"
DATA_URL = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"

DATA_ROOT.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_ROOT)
    print("Done.")
else:
    print("Dataset already exists:", DATA_DIR)

Extracting...
Done.


In [ ]:
class PennFudanDataset(torch.utils.data.Dataset):
    def __init__(self, root, transforms=None):
        self.root = Path(root)
        self.transforms = transforms
        self.imgs = sorted((self.root / "PNGImages").glob("*.png"))
        self.masks = sorted((self.root / "PedMasks").glob("*.png"))

    def __getitem__(self, idx):
        img_path = self.imgs[idx]
        mask_path = self.masks[idx]

        img = torchvision.io.read_image(str(img_path)).float() / 255.0
        mask = torchvision.io.read_image(str(mask_path))[0]

        obj_ids = torch.unique(mask)
        obj_ids = obj_ids[obj_ids != 0]

        masks = mask == obj_ids[:, None, None]
        num_objs = len(obj_ids)

        boxes = []
        for i in range(num_objs):
            pos = torch.where(masks[i])
            xmin = torch.min(pos[1]).item()
            xmax = torch.max(pos[1]).item()
            ymin = torch.min(pos[0]).item()
            ymax = torch.max(pos[0]).item()
            boxes.append([xmin, ymin, xmax, ymax])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.ones((num_objs,), dtype=torch.int64)
        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        iscrowd = torch.zeros((num_objs,), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd,
        }

        if self.transforms is not None:
            img, target = self.transforms(img, target)

        return img, target

    def __len__(self):
        return len(self.imgs)


class ComposeDet:
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, image, target):
        for t in self.transforms:
            image, target = t(image, target)
        return image, target


class ToTensorDet:
    def __call__(self, image, target):
        return image, target


class RandomHorizontalFlipDet:
    def __init__(self, prob=0.5):
        self.prob = prob

    def __call__(self, image, target):
        if torch.rand(1).item() < self.prob:
            image = torch.flip(image, dims=[2])
            w = image.shape[-1]
            boxes = target["boxes"].clone()
            boxes[:, [0, 2]] = w - boxes[:, [2, 0]]
            target["boxes"] = boxes
        return image, target


def get_transform(train):
    transforms = [ToTensorDet()]
    if train:
        transforms.append(RandomHorizontalFlipDet(0.5))
    return ComposeDet(transforms)


def collate_fn(batch):
    return tuple(zip(*batch))


dataset_full = PennFudanDataset(DATA_DIR, transforms=get_transform(train=True))
print("Total images:", len(dataset_full))

Total images: 170


In [ ]:
# Train / val / test split
indices = torch.randperm(len(dataset_full)).tolist()
n_total = len(indices)
n_train = int(TRAIN_RATIO * n_total)
n_val = int(VAL_RATIO * n_total)

train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

train_dataset = PennFudanDataset(DATA_DIR, transforms=get_transform(train=True))
val_dataset = PennFudanDataset(DATA_DIR, transforms=get_transform(train=False))
test_dataset = PennFudanDataset(DATA_DIR, transforms=get_transform(train=False))

train_dataset = Subset(train_dataset, train_idx)
val_dataset = Subset(val_dataset, val_idx)
test_dataset = Subset(test_dataset, test_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False,
                        num_workers=NUM_WORKERS, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False,
                         num_workers=NUM_WORKERS, collate_fn=collate_fn)

len(train_dataset), len(val_dataset), len(test_dataset)

(118, 25, 27)

In [ ]:
# SE building blocks
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(channels, hidden, bias=True)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(hidden, channels, bias=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.shape
        y = self.pool(x).view(b, c)
        y = self.fc1(y)
        y = self.relu(y)
        y = self.fc2(y)
        y = self.sigmoid(y).view(b, c, 1, 1)
        return x * y


class SEBottleneck(Bottleneck):
    def __init__(self, *args, reduction=16, **kwargs):
        super().__init__(*args, **kwargs)
        self.se = SEBlock(self.conv3.out_channels, reduction=reduction)

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)
        out = self.se(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out


def init_se_identity(model):
    for m in model.modules():
        if isinstance(m, SEBlock):
            nn.init.zeros_(m.fc2.weight)
            nn.init.ones_(m.fc2.bias)

In [ ]:
def build_backbone(use_se=False, pretrained=True):
    if use_se:
        backbone_body = ResNet(SEBottleneck, [3, 4, 6, 3])
        init_se_identity(backbone_body)
        if pretrained:
            ref = resnet50(weights=ResNet50_Weights.DEFAULT)
            backbone_body.load_state_dict(ref.state_dict(), strict=False)
            init_se_identity(backbone_body)
    else:
        if pretrained:
            backbone_body = resnet50(weights=ResNet50_Weights.DEFAULT)
        else:
            backbone_body = resnet50(weights=None)

    return_layers = {"layer1": "0", "layer2": "1", "layer3": "2", "layer4": "3"}
    in_channels_list = [256, 512, 1024, 2048]
    out_channels = 256

    backbone = BackboneWithFPN(
        backbone_body,
        return_layers=return_layers,
        in_channels_list=in_channels_list,
        out_channels=out_channels,
    )
    return backbone


def build_detector(use_se=False, pretrained_backbone=True, num_classes=2):
    backbone = build_backbone(use_se=use_se, pretrained=pretrained_backbone)

    anchor_generator = AnchorGenerator(
        sizes=((32,), (64,), (128,), (256,), (512,)),
        aspect_ratios=((0.5, 1.0, 2.0),) * 5,
    )

    roi_pooler = MultiScaleRoIAlign(
        featmap_names=["0", "1", "2", "3", "pool"],
        output_size=7,
        sampling_ratio=2,
    )

    model = FasterRCNN(
        backbone=backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=roi_pooler,
    )
    return model


baseline_model = build_detector(use_se=False, pretrained_backbone=True, num_classes=2).to(device)
se_model = build_detector(use_se=True, pretrained_backbone=True, num_classes=2).to(device)

print("Baseline model ready.")
print("SE model ready.")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 71.7MB/s]


Baseline model ready.
SE model ready.


In [ ]:
def train_one_epoch(model, optimizer, loader, device):
    model.train()
    running_loss = 0.0

    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        running_loss += losses.item()

    return running_loss / max(len(loader), 1)


@torch.no_grad()
def compute_precision_recall(preds, targets, iou_thresh=0.5, score_thresh=0.05):
    tp = fp = fn = 0

    for pred, target in zip(preds, targets):
        pred_boxes = pred["boxes"]
        pred_scores = pred["scores"]

        keep = pred_scores >= score_thresh
        pred_boxes = pred_boxes[keep]
        pred_scores = pred_scores[keep]

        gt_boxes = target["boxes"]

        if len(gt_boxes) == 0 and len(pred_boxes) == 0:
            continue
        if len(gt_boxes) == 0:
            fp += len(pred_boxes)
            continue
        if len(pred_boxes) == 0:
            fn += len(gt_boxes)
            continue

        order = torch.argsort(pred_scores, descending=True)
        pred_boxes = pred_boxes[order]

        ious = box_iou(pred_boxes, gt_boxes)
        matched_gt = set()

        for i in range(pred_boxes.shape[0]):
            best_iou, best_j = torch.max(ious[i], dim=0)
            best_j = int(best_j.item())
            if best_iou.item() >= iou_thresh and best_j not in matched_gt:
                tp += 1
                matched_gt.add(best_j)
            else:
                fp += 1

        fn += (len(gt_boxes) - len(matched_gt))

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    return precision, recall


@torch.no_grad()
def evaluate_model(model, loader, device, score_thresh=0.05):
    model.eval()
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
    preds_all = []
    targets_all = []

    for images, targets in loader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        preds = []
        targs = []

        for o, t in zip(outputs, targets):
            preds.append({
                "boxes": o["boxes"].detach().cpu(),
                "scores": o["scores"].detach().cpu(),
                "labels": o["labels"].detach().cpu(),
            })
            targs.append({
                "boxes": t["boxes"].detach().cpu(),
                "labels": t["labels"].detach().cpu(),
            })

        metric.update(preds, targs)
        preds_all.extend(preds)
        targets_all.extend(targs)

    results = metric.compute()
    precision, recall = compute_precision_recall(preds_all, targets_all, iou_thresh=0.5, score_thresh=score_thresh)

    return {
        "precision": float(precision),
        "recall": float(recall),
        "map_50": float(results["map_50"]),
        "map_5095": float(results["map"]),
        "mar_100": float(results["mar_100"]),
    }


@torch.no_grad()
def benchmark_inference(model, loader, device, warmup=1):
    model.eval()

    for i, (images, _) in enumerate(loader):
        if i >= warmup:
            break
        images = [img.to(device) for img in images]
        _ = model(images)

    start = time.perf_counter()
    n_images = 0

    for images, _ in loader:
        images = [img.to(device) for img in images]
        _ = model(images)
        n_images += len(images)

    elapsed = time.perf_counter() - start
    seconds_per_image = elapsed / max(n_images, 1)
    fps = n_images / max(elapsed, 1e-9)

    return {
        "inference_time_sec": elapsed,
        "seconds_per_image": seconds_per_image,
        "fps": fps,
    }


def train_and_evaluate(model, name, train_loader, test_loader, device, num_epochs=10):
    optimizer = torch.optim.SGD(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

    history = {"train_loss": [], "epoch_time_sec": []}

    print(f"Training {name}...")
    total_start = time.perf_counter()

    for epoch in range(num_epochs):
        epoch_start = time.perf_counter()
        loss = train_one_epoch(model, optimizer, train_loader, device)
        lr_scheduler.step()
        epoch_time = time.perf_counter() - epoch_start

        history["train_loss"].append(loss)
        history["epoch_time_sec"].append(epoch_time)
        print(f"Epoch {epoch+1:02d}/{num_epochs} | loss={loss:.4f} | time={epoch_time:.1f}s")

    train_time = time.perf_counter() - total_start

    print(f"Evaluating {name}...")
    metrics = evaluate_model(model, test_loader, device, score_thresh=SCORE_THRESH)
    speed = benchmark_inference(model, test_loader, device)

    results = {
        "model": name,
        "train_time_sec": train_time,
        "avg_epoch_time_sec": float(np.mean(history["epoch_time_sec"])),
        **metrics,
        **speed,
        "history": history,
    }
    return results

In [ ]:
# Train and evaluate the baseline
baseline_results = train_and_evaluate(
    baseline_model,
    name="Baseline Faster R-CNN",
    train_loader=train_loader,
    test_loader=test_loader,
    device=device,
    num_epochs=NUM_EPOCHS,
)

baseline_results

Training Baseline Faster R-CNN...
Epoch 01/10 | loss=0.4048 | time=41.0s
Epoch 02/10 | loss=0.3313 | time=40.5s
Epoch 03/10 | loss=0.2808 | time=46.7s
Epoch 04/10 | loss=0.2482 | time=47.4s
Epoch 05/10 | loss=0.2492 | time=46.5s
Epoch 06/10 | loss=0.2385 | time=46.4s
Epoch 07/10 | loss=0.2317 | time=46.2s
Epoch 08/10 | loss=0.2336 | time=46.5s
Epoch 09/10 | loss=0.2360 | time=47.0s
Epoch 10/10 | loss=0.2342 | time=46.8s
Evaluating Baseline Faster R-CNN...


{'model': 'Baseline Faster R-CNN',
 'train_time_sec': 454.84463543900006,
 'avg_epoch_time_sec': 45.48418096859998,
 'precision': 0.061652281134325956,
 'recall': 0.892857142841199,
 'map_50': 0.507910430431366,
 'map_5095': 0.1945137232542038,
 'mar_100': 0.3839285671710968,
 'inference_time_sec': 3.4321834810000382,
 'seconds_per_image': 0.12711790670370512,
 'fps': 7.866712298298513,
 'history': {'train_loss': [0.40482653058686496,
   0.3312939021546962,
   0.28084726555872774,
   0.24820360276153533,
   0.2491698738629535,
   0.23850186598502984,
   0.23165582057278036,
   0.23357686349901102,
   0.23603840525877678,
   0.23421937050455707],
  'epoch_time_sec': [40.96196696700008,
   40.533597955000005,
   46.65563995000002,
   47.41771248400005,
   46.46276721300001,
   46.363479052,
   46.194177469999886,
   46.47389073299996,
   46.97543856099992,
   46.803139300999874]}}

In [ ]:
# Train and evaluate the SE-enhanced model
se_results = train_and_evaluate(
    se_model,
    name="SE-Faster R-CNN",
    train_loader=train_loader,
    test_loader=test_loader,
    device=device,
    num_epochs=NUM_EPOCHS,
)

se_results

Training SE-Faster R-CNN...
Epoch 01/10 | loss=0.4176 | time=49.5s
Epoch 02/10 | loss=0.3328 | time=49.1s
Epoch 03/10 | loss=0.3087 | time=49.6s
Epoch 04/10 | loss=0.2773 | time=49.2s
Epoch 05/10 | loss=0.2798 | time=49.3s
Epoch 06/10 | loss=0.2675 | time=50.3s
Epoch 07/10 | loss=0.2651 | time=49.2s
Epoch 08/10 | loss=0.2682 | time=48.7s
Epoch 09/10 | loss=0.2681 | time=49.6s
Epoch 10/10 | loss=0.2634 | time=50.2s
Evaluating SE-Faster R-CNN...


{'model': 'SE-Faster R-CNN',
 'train_time_sec': 494.78828784899997,
 'avg_epoch_time_sec': 49.478494052400016,
 'precision': 0.05953827460503094,
 'recall': 0.874999999984375,
 'map_50': 0.4549613893032074,
 'map_5095': 0.1582237184047699,
 'mar_100': 0.375,
 'inference_time_sec': 3.585740022000209,
 'seconds_per_image': 0.13280518600000774,
 'fps': 7.529826433132977,
 'history': {'train_loss': [0.41761756069579364,
   0.33277775675563487,
   0.30868102509086415,
   0.27729734462701666,
   0.2798036080548319,
   0.26745442490456467,
   0.2650640527323141,
   0.26816366107787115,
   0.268059566490731,
   0.2633508354172868],
  'epoch_time_sec': [49.525230974999886,
   49.107967502000065,
   49.62262268099994,
   49.19833861300003,
   49.3409331360001,
   50.29319399199994,
   49.23605657799999,
   48.6993825049999,
   49.56406611300008,
   50.19714842900021]}}

In [ ]:
# Comparison table
summary = pd.DataFrame([
    {
        "Model": baseline_results["model"],
        "Precision": baseline_results["precision"],
        "Recall": baseline_results["recall"],
        "mAP@0.5": baseline_results["map_50"],
        "mAP@0.5:0.95": baseline_results["map_5095"],
        "Train Time (s)": baseline_results["train_time_sec"],
        "Inference FPS": baseline_results["fps"],
    },
    {
        "Model": se_results["model"],
        "Precision": se_results["precision"],
        "Recall": se_results["recall"],
        "mAP@0.5": se_results["map_50"],
        "mAP@0.5:0.95": se_results["map_5095"],
        "Train Time (s)": se_results["train_time_sec"],
        "Inference FPS": se_results["fps"],
    },
])

summary

,Model,Precision,Recall,mAP@0.5,mAP@0.5:0.95,Train Time (s),Inference FPS
0,Baseline Faster R-CNN,0.061652,0.892857,0.507910,0.194514,454.844635,7.866712
1,SE-Faster R-CNN,0.059538,0.875000,0.454961,0.158224,494.788288,7.529826


## Results Analysis and Report

---

### Experimental Results

The following results were obtained after training both models for **10 epochs** on the Penn-Fudan Pedestrian dataset using SGD (lr=0.005, momentum=0.9, weight decay=5e-4) with a StepLR scheduler (step=3, γ=0.1).

| Metric            | Baseline Faster R-CNN | SE-Faster R-CNN | Difference |
|-------------------|-----------------------|-----------------|------------|
| Precision         | 0.0617                | 0.0596          | −0.0021    |
| Recall            | 0.8929                | 0.8750          | −0.0179    |
| **mAP@0.5**       | **0.5079**            | **0.4550**      | **−0.0529**|
| mAP@0.5:0.95      | 0.1945                | 0.1582          | −0.0363    |
| MAR@100           | 0.3839                | 0.3750          | −0.0089    |
| Train Time (s)    | 454.8                 | 494.8           | +40.0 (+8.8%)|
| Inference FPS     | 7.87                  | 7.53            | −0.34 (−4.3%)|

---

### 1. Did the SE Block Improve Detection Performance?

**No — in this experiment, the baseline outperformed the SE model across all metrics.**

- mAP@0.5 dropped from **0.508 → 0.455** (−5.3 points)
- mAP@0.5:0.95 dropped from **0.195 → 0.158** (−3.6 points)
- Recall dropped slightly from **0.893 → 0.875**

This is a valid and important result. It tells us that SE attention is not universally beneficial — its effectiveness depends on dataset size, class diversity, and training budget.

#### Why did SE underperform here?

1. **Dataset is too simple:** Penn-Fudan has only one foreground class (person) against relatively clean backgrounds. The baseline network has no trouble distinguishing channels — SE has nothing meaningful to recalibrate.
2. **Small dataset (170 images):** SE introduces additional learnable parameters per bottleneck. With limited data, these extra parameters may overfit or fail to converge to useful attention weights within 10 epochs.
3. **Pretrained weights mismatch:** SE block weights are initialized with identity-like outputs (`fc2.weight=0`, `fc2.bias=1`), but the pretrained backbone weights were designed without SE — the two may not co-adapt well in a short training run.
4. **Training epochs:** SE-ResNet models typically require longer training to fully benefit from channel recalibration compared to vanilla ResNets.

---

### 2. Training Loss Comparison

Both models converged smoothly over 10 epochs:

| Epoch | Baseline Loss | SE Model Loss |
|-------|---------------|---------------|
| 1     | 0.4048        | 0.4176        |
| 3     | 0.2808        | 0.3087        |
| 5     | 0.2492        | 0.2798        |
| 7     | 0.2317        | 0.2651        |
| 10    | 0.2342        | 0.2634        |

The SE model consistently had slightly **higher training loss**, suggesting it is harder to optimize. The loss curves are both decreasing monotonically, but the SE model has not fully converged.

---

### 3. How the SE Block Works

The SE block performs **channel-wise feature recalibration** in three steps:

#### Step 1 — Squeeze
Global Average Pooling collapses each channel's spatial feature map into a single scalar:

```
x: [B, C, H, W]  →  z: [B, C]    (one value per channel)
```

#### Step 2 — Excitation
A two-layer fully-connected bottleneck learns which channels matter:

```
s = sigmoid( W2 · ReLU( W1 · z ) )
```

- W1: [C → C/r]  (reduction ratio r=16)
- W2: [C/r → C]
- s: [B, C] — channel importance scores in [0, 1]

#### Step 3 — Scale
Each channel is multiplied by its learned importance score:

```
x̃ = x · s.view(B, C, 1, 1)
```

Channels that are useful for the task receive scores close to 1; noisy or irrelevant channels receive scores close to 0.

---

### 4. Computational Cost Analysis

| Cost Factor         | Baseline | SE Model | Overhead |
|---------------------|----------|----------|----------|
| Avg epoch time (s)  | 45.5     | 49.5     | +8.8%    |
| Total train time (s)| 454.8    | 494.8    | +8.8%    |
| Inference FPS       | 7.87     | 7.53     | −4.3%    |
| Extra params (est.) | —        | ~2.5M    | small    |

SE adds two FC layers per Bottleneck block. ResNet-50 has 16 Bottleneck blocks, so:
- SE parameters ≈ 16 × (2 × C × C/16) — lightweight relative to the full backbone (~25M params)
- The main cost is the additional forward/backward pass through the FC layers and the extra memory for storing intermediate activations

---

### 5. When Does SE Attention Actually Help?

SE tends to improve performance when:

| Scenario | Why SE helps |
|----------|-------------|
| Multi-class detection (e.g., COCO 80 classes) | Different classes activate different channels; SE suppresses irrelevant ones |
| Cluttered or complex backgrounds | Background channels get down-weighted |
| Fine-grained discrimination | Subtle texture/color channels receive higher weight |
| Larger datasets with more training | SE weights have enough examples to converge to meaningful patterns |
| Longer training schedules | SE co-adaptation with backbone requires more epochs |

For **single-class detection on a small, clean dataset** (like this experiment), SE provides little benefit and can even hurt by introducing optimization difficulty.

---

### 6. Advantages of SE Attention

1. **Lightweight:** adds only ~2.5M parameters to ResNet-50 (~10% increase)
2. **Plug-and-play:** can be inserted into any CNN without changing the overall architecture
3. **Interpretable:** the learned scale vector `s` shows which channels the model finds important
4. **No spatial overhead:** only operates channel-wise — no attention maps to compute
5. **Compatible with FPN:** backbone improvements propagate through all FPN levels
6. **Well-studied:** proven gains on ImageNet classification and COCO detection with proper training setup

---

### 7. Limitations of SE Attention

1. **No spatial awareness:** SE only answers *which channel*, not *where in the image*; CBAM and spatial transformers address this
2. **Diminishing returns on simple tasks:** as observed here, single-class detection with clean background sees no gain
3. **Longer convergence:** SE models generally need more epochs to fully co-adapt
4. **Reduction ratio is a hyperparameter:** r=16 is standard but may not be optimal for all architectures
5. **Does not help with scale variation:** FPN handles multi-scale; SE operates per-channel only
6. **Small dataset risk:** extra parameters can overfit or fail to learn meaningful weights

---

### 8. Conclusion

> In this experiment, the **Baseline Faster R-CNN outperformed the SE-augmented model** on the Penn-Fudan Pedestrian dataset (mAP@0.5: 0.508 vs 0.455). This result highlights an important nuance: SE channel attention is not universally beneficial. Its advantages are most pronounced in complex, multi-class detection tasks on large datasets with longer training schedules. On a small, single-class dataset trained for only 10 epochs, the additional complexity of SE blocks introduces optimization challenges without a compensating accuracy gain. The SE model also incurred an 8.8% increase in training time and a 4.3% drop in inference FPS. Future work could investigate (1) training for more epochs, (2) applying SE only to specific ResNet layers, or (3) using CBAM (which adds spatial attention alongside channel attention) to see if broader attention mechanisms provide more robust gains on this dataset.

In [ ]:
## References and Further Reading

1. **Hu, J., Shen, L., & Sun, G. (2018).** Squeeze-and-Excitation Networks. *CVPR 2018.* [[paper]](https://arxiv.org/abs/1709.01507)
   - Original SE paper. Demonstrates 1–2% top-1 accuracy gains on ImageNet with minimal parameter overhead.

2. **Ren, S., He, K., Girshick, R., & Sun, J. (2015).** Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks. *NeurIPS 2015.*
   - Introduces the two-stage detector used as the base model in this assignment.

3. **Lin, T. Y., Dollár, P., Girshick, R., et al. (2017).** Feature Pyramid Networks for Object Detection. *CVPR 2017.*
   - Describes the FPN backbone used to generate multi-scale feature maps.

4. **Woo, S., Park, J., Lee, J. Y., & Kweon, I. S. (2018).** CBAM: Convolutional Block Attention Module. *ECCV 2018.*
   - Extends SE with spatial attention — a natural next step beyond the SE-only approach explored here.

5. **He, K., Zhang, X., Ren, S., & Sun, J. (2016).** Deep Residual Learning for Image Recognition. *CVPR 2016.*
   - ResNet-50 backbone used in this experiment.

---

## Hyperparameter Reference

| Parameter       | Value  | Rationale |
|-----------------|--------|-----------|
| Optimizer       | SGD    | Standard for detection; better generalization than Adam |
| Learning rate   | 0.005  | Scaled down from 0.02 (full COCO batch=16); smaller dataset |
| Momentum        | 0.9    | Standard SGD momentum |
| Weight decay    | 5e-4   | L2 regularization to prevent overfitting |
| LR schedule     | StepLR, step=3, γ=0.1 | Decay LR by 10× every 3 epochs |
| Batch size      | 2      | Memory constraint on single GPU |
| Epochs          | 10     | Sufficient for convergence on small dataset |
| SE reduction r  | 16     | Standard from original paper; balances params vs. capacity |
| Score threshold | 0.05   | Low threshold to capture most proposals for fair evaluation |
| IoU threshold   | 0.5    | Standard PASCAL VOC criterion |
| Train/Val/Test  | 70/15/15% | Standard split; ensures held-out test set |
| Random seed     | 42     | Fixed for reproducibility |